In [1]:
from pathlib import Path
import pymupdf, pandas as pd

reports = Path("../data/reports")
out = Path("../data/extracted")

texts_orig = {pdf.stem: [p.get_text() for p in pymupdf.open(pdf)]
              for pdf in sorted(reports.glob("*.pdf"))}
texts = {k: [t.lower() for t in v] for k, v in texts_orig.items()}

strong_terms = ["gross scope 1", "e1-6", "total ghg emissions", "scope 1 ghg",
                "crude steel", "raw steel", "emissions intensity", "emission intensity",
                "ghg intensity", "co2e per", "per tonne of", "per ton of"]

selected_v2 = {}
for name, pages in texts.items():
    scored = sorted(((sum(t.count(term) for term in strong_terms), i + 1)
                     for i, t in enumerate(pages)), reverse=True)
    selected_v2[name] = sorted(p for s, p in scored[:15] if s >= 1)
    print(f"{name}: {len(pages)} pages, selected {selected_v2[name]}")

2025-26-annual-report: 476 pages, selected [76, 107, 154, 158, 186, 192, 197, 198, 200, 201, 217, 222, 223, 224, 236]
2025-sd-report-22-april-2026-final-upload: 94 pages, selected [24, 32, 50, 51, 58, 59, 60, 63, 65, 66, 67, 75, 83, 84, 87]
ESG Factsheet: 16 pages, selected [1, 2, 3, 4, 11, 12]
Nucor_10K_2025: 124 pages, selected [9, 10, 11, 14, 16]
Nucor_SASB_index: 9 pages, selected [3, 4, 5]
POSCO_Sustainability_Report_2024_eng: 193 pages, selected [17, 157, 158, 159, 160, 161, 162, 172, 173, 175, 176, 189]
SSAB_AR_2025: 189 pages, selected [7, 17, 26, 28, 30, 53, 67, 79, 80, 83, 84, 85, 87, 115, 187]
keydata-ar-2025-en: 2 pages, selected [1]
szag-esg-figures-2025-en: 19 pages, selected [2, 4, 5, 8]
thyssenkrupp_AR_2024-25: 347 pages, selected [35, 52, 69, 70, 73, 85, 90, 113, 114, 118, 119, 120, 123, 124]


In [2]:
import os, time
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel

load_dotenv("../.env", override=True)
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"], http_options={"timeout": 120000})
MODEL = "gemini-3.6-flash"

class Value(BaseModel):
    value: float | None
    unit: str | None
    year: str | None
    page: int | None
    quote: str | None

class Extraction(BaseModel):
    crude_steel_production: Value
    scope1_emissions: Value
    scope2_emissions: Value
    co2_intensity: Value
    steelmaking_capacity: Value

PROMPT_V2 = """You extract reported figures from a steel company's annual report or sustainability disclosure.
Use ONLY the pages below.

Rules:
1. Take the value for the most recent reporting year shown. Tables often have several columns
   (base year, previous year, reporting year, targets). Use the reporting-year column.
   Never return a base-year value, a target or a forecast.
2. Copy the number exactly as printed. Do not convert units or calculate anything.
3. Always give the unit as printed (e.g. "Mt CO2e", "thousand tonnes", "t CO2e/t crude steel").
   If the unit is only in a table or column header, take it from there.
4. Give the reporting year as printed (e.g. "2025" or "2024/2025"), the page number from the page marker,
   and a short quote (max 20 words) containing the number.
5. Use group-wide totals, not single sites or segments.
6. If a field is not on these pages, return null for all its parts. Do not guess.

Fields:
- crude_steel_production: total crude steel produced in the reporting year (US reports may say "raw steel" or "tons cast")
- scope1_emissions: total gross Scope 1 GHG emissions
- scope2_emissions: total gross Scope 2 GHG emissions, market-based if both are given
- co2_intensity: GHG or CO2 emissions per tonne of steel (not per revenue)
- steelmaking_capacity: total crude steelmaking capacity per year

PAGES:
"""

def run_extraction(companies, prompt, selection, version, tries=3):
    rows = []
    for company, docs in companies.items():
        pages = "\n\n".join(f"--- DOCUMENT {d}, PAGE {p} ---\n{texts_orig[d][p - 1]}"
                            for d in docs for p in selection[d])
        for attempt in range(1, tries + 1):
            try:
                response = client.models.generate_content(
                    model=MODEL, contents=prompt + pages,
                    config={"response_mime_type": "application/json",
                            "response_schema": Extraction, "temperature": 0})
                break
            except Exception as e:
                print(f"{company}: attempt {attempt} failed ({str(e)[:60]})")
                time.sleep(15 * attempt)
        else:
            print(f"{company}: SKIPPED")
            continue
        for field, v in response.parsed.model_dump().items():
            rows.append({"company": company, "field": field, **v, "model": MODEL, "prompt": version})
        print(f"{company}: done")
    return pd.DataFrame(rows)

print("Setup OK, key length:", len(os.environ.get("GEMINI_API_KEY", "")))

Setup OK, key length: 53


In [3]:
MODEL = "gemini-3.5-flash"

In [ ]:
new_companies = {
    "Salzgitter": ["szag-esg-figures-2025-en", "keydata-ar-2025-en"],
    "Tata Steel": ["ESG Factsheet"],
    "voestalpine": ["2025-26-annual-report"],
    "ArcelorMittal": ["2025-sd-report-22-april-2026-final-upload"],
    "POSCO": ["POSCO_Sustainability_Report_2024_eng"],
}
df_new = run_extraction(new_companies, PROMPT_V2, selected_v2, "v2")
df_new.to_csv(out / "extractions_v2_batch2.csv", index=False)
print(df_new[["company", "field", "value", "unit", "year", "page"]].to_string(index=False))

Salzgitter: attempt 1 failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This mo)
Salzgitter: attempt 2 failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This mo)
Salzgitter: attempt 3 failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This mo)


In [ ]:
test_models = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash",
               "gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]
for m in test_models:
    try:
        r = client.models.generate_content(model=m, contents="Reply with just: OK")
        print(m, "->", r.text.strip())
    except Exception as e:
        print(m, "-> ERROR", str(e)[:40])